# 03 - Ingestão dos preços de combustíveis da ANP

## Objetivo
Realizar a ingestão dos arquivos mensais de preços de combustíveis disponibilizados pela Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP), preservando os dados brutos na camada Bronze do Lakehouse.

## Origem
Arquivos CSV mensais armazenados no Volume RAW do Databricks.

## Escopo
Dados de preços de gasolina, etanol, diesel e GNV.

## Destino
Tabelas da camada `workspace.bronze`.

In [0]:
caminho_anp = "/Volumes/workspace/raw/dados_raw/anp_precos/"

print(caminho_anp)

/Volumes/workspace/raw/dados_raw/anp_precos/


## Inspeção dos arquivos de origem

Os arquivos de preços da ANP são disponibilizados em dois conjuntos: **Diesel/GNV** e **Gasolina/Etanol**.

Antes da leitura conjunta de todos os CSVs, é realizada uma inspeção de arquivos representativos de cada conjunto para verificar seus campos, tipos de dados e compatibilidade estrutural.

Essa verificação permite confirmar se os arquivos podem ser processados em uma única etapa de ingestão para a camada Bronze.

In [0]:
arquivo_teste = (
    "/Volumes/workspace/raw/dados_raw/anp_precos/"
    "01-dados-abertos-precos-diesel-gnv.csv"
)

df_teste = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .csv(arquivo_teste)
)

df_teste.printSchema()

root
 |-- Regiao - Sigla: string (nullable = true)
 |-- Estado - Sigla: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Revenda: string (nullable = true)
 |-- CNPJ da Revenda: string (nullable = true)
 |-- Nome da Rua: string (nullable = true)
 |-- Numero Rua: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cep: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Data da Coleta: date (nullable = true)
 |-- Valor de Venda: string (nullable = true)
 |-- Valor de Compra: string (nullable = true)
 |-- Unidade de Medida: string (nullable = true)
 |-- Bandeira: string (nullable = true)



In [0]:
display(df_teste.limit(10))

Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
NE,BA,VALENCA,ORGANIZACAO GUAIBIM LTDA,01.878.111/0001-68,AVENIDA ANTONIO C MAGALHAES,500,CASA,SAO FELIX,45400-000,DIESEL,2026-01-01,"5,99",null,R$ / litro,RAIZEN
N,TO,PARAISO DO TOCANTINS,MEDEIROS COMERCIO VAREJISTA DE COMBUSTIVEIS LTDA,03.775.225/0001-08,AVENIDA TRANSBRASILIANA,961,CX. POSTAL 23,CENTRO,77600-000,DIESEL,2026-01-01,"5,99",null,R$ / litro,BRANCA
N,TO,PARAISO DO TOCANTINS,MEDEIROS COMERCIO VAREJISTA DE COMBUSTIVEIS LTDA,03.775.225/0001-08,AVENIDA TRANSBRASILIANA,961,CX. POSTAL 23,CENTRO,77600-000,DIESEL S10,2026-01-01,"5,99",null,R$ / litro,BRANCA
N,TO,PARAISO DO TOCANTINS,LOPES & MARINHO LTDA,01.066.091/0001-20,AVENIDA CASTELO BRANCO,1111,QUADRA21 LOTE 7A LOTE 8 LOTE 9 LOTE 10,CENTRO,77600-000,DIESEL,2026-01-01,"5,89",null,R$ / litro,VIBRA
N,TO,PARAISO DO TOCANTINS,LOPES & MARINHO LTDA,01.066.091/0001-20,AVENIDA CASTELO BRANCO,1111,QUADRA21 LOTE 7A LOTE 8 LOTE 9 LOTE 10,CENTRO,77600-000,DIESEL S10,2026-01-01,"5,89",null,R$ / litro,VIBRA
SE,MG,GUAXUPE,SAO PAULO MINAS COMERCIO DERIVADOS DE PETROLEO LTDA.,05.282.048/0002-34,AVENIDA FELIPE ELIAS ZEITUNE,130,null,CENTRO,37800-000,DIESEL,2026-01-01,"5,89",null,R$ / litro,BRANCA
SE,MG,GUAXUPE,SAO PAULO MINAS COMERCIO DERIVADOS DE PETROLEO LTDA.,05.282.048/0002-34,AVENIDA FELIPE ELIAS ZEITUNE,130,null,CENTRO,37800-000,DIESEL S10,2026-01-01,"5,99",null,R$ / litro,BRANCA
SE,MG,PARA DE MINAS,AUTO POSTO IRMAOS MELGACO LTDA,08.538.103/0001-11,AVENIDA PRESIDENTE VARGAS,500,null,DONA TUNICA,35661-000,DIESEL,2026-01-01,"5,75",null,R$ / litro,VIBRA
SE,MG,PARA DE MINAS,AUTO POSTO IRMAOS MELGACO LTDA,08.538.103/0001-11,AVENIDA PRESIDENTE VARGAS,500,null,DONA TUNICA,35661-000,DIESEL S10,2026-01-01,"5,85",null,R$ / litro,VIBRA
SE,MG,PARA DE MINAS,AUTO POSTO MR LTDA,05.835.706/0001-97,AVENIDA PROFESSOR MELLO CANCADO,521,null,SAO JOSE,35660-573,DIESEL,2026-01-01,"6,14",null,R$ / litro,IPIRANGA


In [0]:
arquivo_teste_gasolina = (
    "/Volumes/workspace/raw/dados_raw/anp_precos/"
    "01-dados-abertos-precos-gasolina-etanol.csv"
)

df_teste_gasolina = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .csv(arquivo_teste_gasolina)
)

df_teste_gasolina.printSchema()

root
 |-- Regiao - Sigla: string (nullable = true)
 |-- Estado - Sigla: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Revenda: string (nullable = true)
 |-- CNPJ da Revenda: string (nullable = true)
 |-- Nome da Rua: string (nullable = true)
 |-- Numero Rua: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cep: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Data da Coleta: date (nullable = true)
 |-- Valor de Venda: string (nullable = true)
 |-- Valor de Compra: string (nullable = true)
 |-- Unidade de Medida: string (nullable = true)
 |-- Bandeira: string (nullable = true)



In [0]:
print("Mesmo número de colunas:",
      len(df_teste.columns) == len(df_teste_gasolina.columns))

print("Mesmos nomes de colunas:",
      df_teste.columns == df_teste_gasolina.columns)

Mesmo número de colunas: True
Mesmos nomes de colunas: True


### Compatibilidade entre os arquivos

A comparação dos arquivos de Diesel/GNV e Gasolina/Etanol mostrou que ambos apresentam a mesma estrutura de colunas.

Com a compatibilidade do schema verificada, os arquivos mensais podem ser lidos conjuntamente pelo Spark, permitindo consolidar as diferentes categorias de combustíveis em um único DataFrame na camada Bronze.

In [0]:
df_anp_raw = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .csv("/Volumes/workspace/raw/dados_raw/anp_precos/*.csv")
)

## Ingestão dos arquivos

Após a verificação de compatibilidade, todos os arquivos CSV disponíveis no diretório são lidos conjuntamente pelo Spark.

O uso do padrão `*.csv` permite consolidar os arquivos mensais em um único DataFrame, mantendo nesta etapa a estrutura original dos dados. Nas etapas seguintes são adicionadas informações de rastreabilidade e realizada a padronização dos nomes das colunas antes da persistência na camada Bronze.

### Metadados de rastreabilidade

Antes da persistência na camada Bronze, são adicionadas informações de rastreabilidade aos registros:

- **Arquivo de origem:** identifica o CSV do qual cada registro foi carregado;
- **Data de ingestão:** registra o momento em que os dados foram processados;
- **Fonte:** identifica a ANP como origem do conjunto de dados.

Esses metadados permitem rastrear os registros até seus arquivos de origem e facilitam futuras verificações de qualidade e auditoria do pipeline.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

df_anp_bronze = (
    df_anp_raw
        .withColumn("_arquivo_origem", col("_metadata.file_path"))
        .withColumn("_data_ingestao", current_timestamp())
        .withColumn("_fonte", lit("ANP"))
)

### Validação da ingestão

Após a consolidação dos arquivos e a inclusão dos metadados de rastreabilidade, são verificados o total de registros carregados e o schema resultante.

Essa conferência permite validar a estrutura do conjunto consolidado antes da padronização dos nomes das colunas e da persistência na camada Bronze.

In [0]:
print(f"Registros carregados: {df_anp_bronze.count()}")
print(f"Colunas: {len(df_anp_bronze.columns)}")

df_anp_bronze.printSchema()

Registros carregados: 1374816
Colunas: 19
root
 |-- Regiao - Sigla: string (nullable = true)
 |-- Estado - Sigla: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Revenda: string (nullable = true)
 |-- CNPJ da Revenda: string (nullable = true)
 |-- Nome da Rua: string (nullable = true)
 |-- Numero Rua: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cep: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Data da Coleta: date (nullable = true)
 |-- Valor de Venda: string (nullable = true)
 |-- Valor de Compra: string (nullable = true)
 |-- Unidade de Medida: string (nullable = true)
 |-- Bandeira: string (nullable = true)
 |-- _arquivo_origem: string (nullable = false)
 |-- _data_ingestao: timestamp (nullable = false)
 |-- _fonte: string (nullable = false)



In [0]:
arquivos_lidos = (
    df_anp_bronze
        .select("_arquivo_origem")
        .distinct()
)

print(f"Arquivos efetivamente ingeridos: {arquivos_lidos.count()}")

Arquivos efetivamente ingeridos: 40


## Padronização dos nomes das colunas

Os arquivos da ANP possuem nomes de campos com espaços, acentos e outros caracteres que podem dificultar sua utilização nas etapas posteriores do pipeline. Para padronizar o schema, é aplicada uma função que remove acentos, substitui caracteres especiais por `_` e elimina underscores duplicados ou posicionados no início e no fim dos nomes.

Essa transformação altera apenas os nomes das colunas, preservando os valores dos registros ingeridos.

In [0]:
import re
import unicodedata

def normalizar_nome_coluna(nome):
    nome = unicodedata.normalize("NFKD", str(nome))
    nome = nome.encode("ascii", "ignore").decode("ascii")
    nome = re.sub(r"[^a-zA-Z0-9_]", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    return nome.strip("_")

In [0]:
df_anp_bronze_normalizado = df_anp_bronze

for coluna in df_anp_bronze.columns:
    novo_nome = normalizar_nome_coluna(coluna)
    df_anp_bronze_normalizado = (
        df_anp_bronze_normalizado
        .withColumnRenamed(coluna, novo_nome)
    )

df_anp_bronze_normalizado.printSchema()

root
 |-- Regiao_Sigla: string (nullable = true)
 |-- Estado_Sigla: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Revenda: string (nullable = true)
 |-- CNPJ_da_Revenda: string (nullable = true)
 |-- Nome_da_Rua: string (nullable = true)
 |-- Numero_Rua: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cep: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Data_da_Coleta: date (nullable = true)
 |-- Valor_de_Venda: string (nullable = true)
 |-- Valor_de_Compra: string (nullable = true)
 |-- Unidade_de_Medida: string (nullable = true)
 |-- Bandeira: string (nullable = true)
 |-- arquivo_origem: string (nullable = false)
 |-- data_ingestao: timestamp (nullable = false)
 |-- fonte: string (nullable = false)



In [0]:
print(f"Colunas após normalização: {len(df_anp_bronze_normalizado.columns)}")
print(df_anp_bronze_normalizado.columns)

Colunas após normalização: 19
['Regiao_Sigla', 'Estado_Sigla', 'Municipio', 'Revenda', 'CNPJ_da_Revenda', 'Nome_da_Rua', 'Numero_Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data_da_Coleta', 'Valor_de_Venda', 'Valor_de_Compra', 'Unidade_de_Medida', 'Bandeira', 'arquivo_origem', 'data_ingestao', 'fonte']


## Persistência na camada Bronze

Após a consolidação dos arquivos, inclusão dos metadados e padronização dos nomes das colunas, o DataFrame é persistido em formato Delta na tabela `workspace.bronze.precos_anp_raw`.

A gravação utiliza substituição controlada do schema, permitindo recriar a tabela com a estrutura definida pelo processo de ingestão. Após a persistência, a tabela é novamente carregada e sua quantidade de registros é comparada à do DataFrame de origem para verificar se houve perda de registros durante a gravação.

In [0]:
(
    df_anp_bronze_normalizado.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.bronze.precos_anp_raw")
)

print("Tabela workspace.bronze.precos_anp_raw criada com sucesso.")

Tabela workspace.bronze.precos_anp_raw criada com sucesso.


In [0]:
df_anp_bronze_salva = spark.table("workspace.bronze.precos_anp_raw")

print(f"Registros na origem: {df_anp_bronze_normalizado.count()}")
print(f"Registros na Bronze: {df_anp_bronze_salva.count()}")

Registros na origem: 1374816
Registros na Bronze: 1374816
